In [1]:
import os
import re
import warnings
import gc
import torch
import pandas as pd
import numpy as np

# NLP & Visuals
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet, stopwords
from nltk import pos_tag
from spacy.lang.en.stop_words import STOP_WORDS as SPACY_STOP_WORDS

# BERTopic & Modeling
from bertopic import BERTopic
from bertopic.representation import BaseRepresentation
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Settings
warnings.filterwarnings('ignore')

# Download necessary NLTK data
nltk_dir = "../../../data/auto/"
os.makedirs(nltk_dir, exist_ok=True)
nltk.data.path.append(nltk_dir)
for pkg in ['punkt', 'punkt_tab', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, download_dir=nltk_dir, quiet=True)

2026-03-22 18:09:38.305353: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# ==========================================
# Loading data
# ==========================================
try:
    df = pd.read_csv("../../../data/policy/combined_policy_docs_chunked.csv", encoding='utf-8-sig')
    df['Title'] = df['Title'].fillna('').astype(str)
    df = df[df['Content'] != 'N/A'].dropna(subset=['Content'])
    df['Content'] = df['Title'] + " " + df['Content'] 
    print(f"Successfully loaded {len(df)} documents.")
except FileNotFoundError:
    print("Error: File not found. Check path.")
    exit()

Successfully loaded 1172 documents.


In [3]:
# ==========================================
# Refined & consolidated stopwords
# ==========================================
stop_words = set(SPACY_STOP_WORDS).union(set(stopwords.words('english')))
try:
    stop_words.update(set(stopwords.words('french')))
except:
    pass

# ==========================================
# Consolidated refined stopwords for BERTopic & SLM
# ==========================================
pdf_structural_noise = [
    'pdf', 'qxp', 'page', 'copyright', 'reserved', 'www', 'http', 'https', 'com', 'org',
    'figure', 'table', 'annex', 'appendix', 'footnote', 'endnote', 'doi', 'isbn', 
    'bibliography', 'citation', 'reference', 'metadata', 'docx', 'executive summary', 
    'draft', 'matrix', 'blueprint', 'snapshot', 'chart', 'brochure', 'section', 'chapter', 
    'part', 'title', 'article', 'clause', 'paragraph', 'subparagraph', 'schedule', 
    'exhibit', 'supplement', 'addendum', 'preamble', 'recital', 'report', 'disposition',
    'alinea', 'expos', 'motifs', 'manual', 'handbook', 'procedure', 'form', 'file', 'save'
]

legal_vague_noise = [
    'shall', 'must', 'may', 'might', 'could', 'would', 'should', 'will', 'pursuant',
    'herein', 'therein', 'notwithstanding', 'accordance', 'furthermore', 'moreover',
    'hereby', 'whereby', 'hereinafter', 'thereafter', 'thereto', 'coppa', 'ferpa', 
    'enact', 'bit', 'actually', 'typical', 'various', 'several', 'final', 'total', 
    'sample', 'original', 'high', 'low', 'pro', 'stu', 'etc', 'august', 'april', 
    'july', 'session', 'use', 'policy', 'policies', 'document', 'documents',
    'framework', 'frameworks', 'regulation', 'regulations', 'law', 'laws', 
    'act', 'acts', 'code', 'codes', 'statute', 'statutes', 'rule', 'rules', 'standard',
    'guidance', 'information', 'guidance', 'guidelines', 'information', 'informations', 
    'data', 'dataset',
]

geo_admin_noise = [
    'ireland', 'irish', 'france', 'french', 'usa', 'america', 'australia', 'australian',
    'country', 'nation', 'national', 'state', 'government', 'federal', 'ministry', 'department',
    'gouvernement', 'ministre', 'dtat', 'republic', 'pittsburg', 'antioch', 'medina', 
    'winterset', 'sara', 'washington', 'california', 'massachusetts', 'oregon', 'sweden', 
    'unified', 'district', 'nsw', 'commonwealth', 'australasian', 'estonia', 'ny', 
    'new york', 'suffolk', 'long island', 'palo', 'alto', 'ca', 'ma', 'wa', 'northport', 
    'nenufsd', 'nenuf', 'sd', 'school district', 'administrative', 'administration', 
    'office', 'clerk', 'board', 'superintendent', 'enrollment', 'attendance', 'going', 'oklahoma'
]

domain_tech_noise = [
    'artificial', 'intelligence', 'ai', 'ia', 'genai', 'generative', 'model', 'models',
    'algorithm', 'algorithms', 'data', 'digital', 'technology', 'technological', 'software',
    'education', 'educational', 'pupil', 'classroom', 'learning', 'cole',
    'teaching', 'instruction', 'curriculum', 'student', 'teacher', 'teachers', 'educatif', 'enseignant',
    'professeur', 'programme', 'apprentissage', 'enseignement', 'formation', 'établissement',
    'covid', 'primary', 'technical', 'platform', 'online', 'internet', 'web', 'system', 'systems',
    'device', 'user', 'login', 'portal', 'dashboard', 'click', 'tab', 'school', 'schools', 'people', 'public',
    'students', 'generated', 'pupils', 'technologies', 'schools', 'districts', 'educators', 
    'educational institutions', 'edtech', 'edutech', 'edtech companies', 'edutech companies', 'edtech sector', 'edutech',
    'élèves', 'usages', 'éducation', 'enseignants', 'établissements', 'apprentissage', 'enseignement', 'formation', 
    'programme', 'pédagogique', 'numérique', 'technologie', 'technologique',
]

org_name_fragment_noise = [
    'adam', 'mary', 'jensen', 'abraham', 'lincoln', 'knight', 'kim', 'guez',
    'oecd', 'cbs', 'webwise', 'cosn', 'dlf', 'scoilnet', 'deap', 'dne', 'men', 
    'parliament', 'commissioner', 'sector', 'cipa', 'protection act', 
    'oide', 'tie', 'cpd', 'pisa', 'ieee', 'openai', 'microsoft', 'google', 'amazon',
    'ing', 'tion', 'ment', 'ness', 'ly', 'able', 'ive', 'ent', 'ant', 'ence', 'ance', 
    'ali', 'gue', 'rod', 'rodr', 'educa', 'evi', 'showcase', 'deliverable', 'post', 
    'fig', 'fee', 'dna', 'yim', 'dai', 'shamir', 'sara', 'guez', 'rod', 'rodr', 'ela',
]

all_custom_stops = (
    pdf_structural_noise + legal_vague_noise + 
    geo_admin_noise + domain_tech_noise + 
    org_name_fragment_noise
)

stop_words.update(all_custom_stops)

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'): return wordnet.ADJ
    elif treebank_tag.startswith('V'): return wordnet.VERB
    elif treebank_tag.startswith('N'): return wordnet.NOUN
    elif treebank_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def clean_text(text):
    # Basic Validation
    if not isinstance(text, str) or len(text) < 5: return ""
    
    # French Apostrophe Handling (Alignment with TfidfVectorizer)
    # Replaces ' with a space (e.g., "l'éducation" -> "l éducation") 
    # so the vectorizer can recognize the core noun as a standalone token.
    text = re.sub(r"([a-zA-ZàâäéèêëïîôùûüÿçÀÂÄÉÈÊËÏÎÔÙÛÜŸÇ])'", r"\1 ", text)
    
    # Stop-Fragment Removal
    # Removes single-letter French articles/prepositions that lose meaning after splitting.
    text = re.sub(r"\b(l|d|c|j|qu)\b", "", text, flags=re.IGNORECASE)
    
    # Normalization & Special Character Stripping
    text = text.lower()
    # Matches the 'token_pattern' in your Vectorizer to keep accented characters 
    # essential for French policy terms (e.g., 'élèves', 'éthique').
    text = re.sub(r'[^a-zàâäéèêëïîôùûüÿç\s]', ' ', text)
    
    # Semantic Lemmatization
    # Uses POS tagging to reduce words to their dictionary root (e.g., 'teaching' -> 'teach').
    words = text.split()
    tagged = pos_tag(words)
    clean_words = [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged]
    
    # Final Filter
    # Removes stop words and ensures all tokens meet the minimum 3-character 
    # length requirement set in the TfidfVectorizer pattern.
    return " ".join([w for w in clean_words if w not in stop_words and len(w) > 2])

In [4]:
# ==========================================
# Representation model (SLM) wrapper
# ==========================================
class LocalSLMWrapper(BaseRepresentation):
    """
    Custom wrapper to integrate a Local SLM (Small Language Model) 
    for automated topic labeling within BERTopic.
    """
    def __init__(self, pipe):
        self.pipeline = pipe

    def extract_topics(self, topic_model, documents, tf_idf, topics):
        topic_labels = {}
        for topic in topics.keys():
            # Handle noise/outliers (Topic -1)
            if topic == -1:
                topic_labels[topic] = ["Outliers"]
                continue
            
            # Extract top 10 keywords from c-TF-IDF to use as LLM prompt context
            words = [word for word, _ in topics[topic][:10]]
            keywords_str = ", ".join(words)
            
            prompt = f"Keywords: {keywords_str}\nPolicy Theme (3 words max):"
            
            try:
                # Generate concise labels using the SLM pipeline
                output = self.pipeline(prompt, max_new_tokens=15, do_sample=False)
                res = output[0]['generated_text'].split("Policy Theme (3 words max):")[-1].strip()
                clean_label = res.split('\n')[0].replace('"', '').strip()
                
                # Hallucination Guardrail: Reverts to top keywords if the LLM output 
                # contains filler phrases or lacks sufficient detail.
                # Expanded Hallucination Guardrail
                bad_patterns = [
                    # Conversational/Definitional Fillers
                    "type of", "is a", "refers to", "consists of", "related to", 
                    "associated with", "example of", "category of", "deals with",
                    
                    # Geographical/Administrative Noise (to avoid hyper-local labeling)
                    "located", "state of", "district", "county", "region of", 
                    "territory", "city of", "department of",
                    
                    # Meta-talk (the LLM describing its own output)
                    "policy theme", "keywords include", "the theme is", "summary:", 
                    "topic involves", "here is", "this is",
                    
                    # Overly Broad/Vague Terms
                    "various", "multiple", "etc", "and so on", "general", "aspects",
                    
                    # Structural/Methodological Noise
                    "interview with", "excerpt from", "document", "source", "page", 
                    "table", "section", "paragraph"
                ]
                if any(p in clean_label.lower() for p in bad_patterns) or len(clean_label) < 3:
                    clean_label = f"{words[0].title()} & {words[1].title()}"
                
                print(f"Topic {topic} Keywords: {keywords_str[:50]}... -> Label: {clean_label}")
                topic_labels[topic] = [clean_label]
            except:
                # Fallback mechanism in case of generation failure
                topic_labels[topic] = [f"{words[0]} {words[1]}"]
                
        return topic_labels

In [5]:
# ==========================================
# Initialize models & components
# ==========================================

# Hardware acceleration check
# Detects if a GPU (CUDA) is available to significantly speed up embedding and 
# generation; defaults to CPU if no dedicated graphics hardware is present.
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Multilingual semantic embedding model
# Loads a transformer model optimized for mapping sentences into a shared vector 
# space. The 'multilingual-MiniLM' is chosen for its balance of speed and 
# accuracy in handling cross-lingual policy themes (e.g., English and French).
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Small Language Model (SLM) configuration
# Utilizes Qwen2-0.5B-Instruct, a compact but powerful model designed to 
# interpret clusters of text and generate human-readable labels.

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
# model_id = "Qwen/Qwen2-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Model loading with memory optimization
# Loads the causal language model with float16 precision (on GPU) to reduce 
# VRAM usage while maintaining the thematic nuance required for policy labeling.
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    dtype=torch.float16 if device=="cuda" else torch.float32
).to(device)

# 5. Pipeline & custom wrapper integration
# Wraps the SLM in a text-generation pipeline and connects it to the 
# LocalSLMWrapper, allowing BERTopic to use 'intelligent' synthesis for 
# topic discovery rather than relying solely on raw keyword lists.
slm_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0 if device=="cuda" else -1)
slm_labeler = LocalSLMWrapper(slm_pipe)

Device set to use cpu


In [6]:
MODELS_DIR = "../../../models/"
OUTPUT_DIR = "../../../data/policy/"
os.makedirs(MODELS_DIR, exist_ok=True)

# ==========================================
# Final per-country analysis loop
# ==========================================
for country in df['Country'].unique():
    print(f"\n{'='*20} Processing: {country.upper()} {'='*20}")
    c_df = df[df['Country'] == country].copy()
    if len(c_df) < 15: continue

    raw_docs = c_df['Content'].tolist()

    # Use the raw data to see what would happen without cleaning, since the vectorizer is set to handle it.
    # cleaned_docs = [clean_text(d) for d in raw_docs]
    embeddings = embedding_model.encode(raw_docs, show_progress_bar=False)

    umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
    hdbscan_model = HDBSCAN(min_cluster_size=10, min_samples=2, metric='euclidean', cluster_selection_method='eom')

    vectorizer_model = TfidfVectorizer(
        ngram_range=(1, 2), 
        stop_words=list(stop_words),
        token_pattern=r"(?u)\b[a-zA-Zàâäéèêëïîôùûüÿç]{3,}\b", 
        min_df=2 
    )

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        representation_model=slm_labeler,
        verbose=True
    )

    topics, _ = topic_model.fit_transform(raw_docs, embeddings)

    topic_info = topic_model.get_topic_info()
    if "Representation" in topic_info.columns:
        topic_info['Topic_Label'] = topic_info['Representation'].apply(lambda x: x[0] if isinstance(x, list) else x)

    c_df['Topic_ID'] = topics
    label_map = dict(zip(topic_info['Topic'], topic_info['Topic_Label']))
    c_df['Topic_Label_Description'] = c_df['Topic_ID'].map(label_map)

    safe_name = country.lower().replace(" ", "_")
    out_path = os.path.join(OUTPUT_DIR, safe_name)
    os.makedirs(out_path, exist_ok=True)
    
    c_df.to_csv(os.path.join(out_path, f"document_assignments_{safe_name}.csv"), index=False, encoding='utf-8-sig')
    summary_df = topic_info.drop(columns=['Representative_Docs'], errors='ignore')
    summary_df.to_csv(os.path.join(out_path, f"summary_{safe_name}.csv"), index=False, encoding='utf-8-sig')

    # Cleanup
    del embeddings, topic_model, c_df
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()

print("\nAll countries processed. Check the output directory policy/<country_name> for results.")


==================== Processing: IRELAND ====================


2026-03-22 18:10:03,819 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-22 18:10:11,696 - BERTopic - Dimensionality - Completed ✓
2026-03-22 18:10:11,696 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-22 18:10:11,714 - BERTopic - Cluster - Completed ✓
2026-03-22 18:10:11,717 - BERTopic - Representation - Fine-tuning topics using representation models.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Topic 0 Keywords: strategy, tools, skills, support, impact, programm... -> Label: Improve the quality of education through innovative and effective strategies.
Topic 1 Keywords: project, cluster, clusters, evaluation, sef, proje... -> Label: Support for the development of a sustainable and inclusive education system.
Topic 2 Keywords: tools, committee, content, ensure, assessment, sta... -> Label: Human Rights and the Committee


2026-03-22 18:10:36,779 - BERTopic - Representation - Completed ✓


Topic 3 Keywords: cyberbullying, bullying, studies, empathy, moral, ... -> Label: Empathy and Social Responsibility

==================== Processing: FRANCE ====================


...

Topic 9 Keywords: administrators affiliates, affiliates, staff admin... -> Label: Staff and Parents/Guardians

==================== Processing: AUSTRALIA ====================


2026-03-22 18:13:22,631 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-22 18:13:22,988 - BERTopic - Dimensionality - Completed ✓
2026-03-22 18:13:22,989 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-22 18:13:22,994 - BERTopic - Cluster - Completed ✓
2026-03-22 18:13:22,996 - BERTopic - Representation - Fine-tuning topics using representation models.


Topic 0 Keywords: principles, principles practice, practice, submiss... -> Label: AI ethics
Topic 1 Keywords: cognitive, offloading, cognitive offloading, offlo... -> Label: Cognitive Offloading
Topic 2 Keywords: tools, respondents, survey, independent, responden... -> Label: Workforce
Topic 3 Keywords: literacy, studies, computational, programming, res... -> Label: Theoretical and Practical Approaches to Literacy Education
Topic 4 Keywords: box, principles, principles practical, principle, ... -> Label: Well-being
Topic 5 Keywords: vsv, employees, staff, tools, responsible, princip... -> Label: Fairness in the Workplace
Topic 6 Keywords: adoption, academic, phase, tertiary, submissions, ... -> Label: Academic and Institutional Support


2026-03-22 18:14:16,901 - BERTopic - Representation - Completed ✓


Topic 7 Keywords: tools, staff, october, sensitive, plagiarism, uplo... -> Label: Confidentiality

All countries processed. Check the output directory policy/<country_name> for results.


## Comparative Analysis: BERTopic + Qwen2.5-0.5B vs. Fine-Grained LDA

The implementation of **BERTopic** with the **Qwen/Qwen2.5-0.5B-Instruct** SLM significantly shifts the analysis from frequency-based clustering (LDA) to semantic representation. While LDA provided a broad "policy DNA," the Qwen-enhanced pipeline extracts **highly specific operational sub-themes**, revealing granular details that LDA aggregated. However, the strict output constraint (2-3 words) combined with "unclean data" has introduced specific **label hallucinations**.

### USA: Deconstructing "Privacy and Equity"
The LDA analysis broadly categorized the US under "Compliance" and "Equity." BERTopic + Qwen successfully deconstructs this into actionable sub-components:
*   **Granularity Gain:** It separates **"Protecting Staff Privacy"** (Topic 6) from general student privacy, identifying a specific focus on employee data rights that LDA missed. It also isolates **"Literacy and the Arts"** (Topic 1), distinguishing the humanities focus from general "achievement."
*   **New Topic - Synthetic Media:** Topic 5 (`synthetic media`, `classrooms`) reveals a distinct conversation about **Deepfakes/Generative AI content**, a critical modern risk absent from the LDA "Governance" topic.
*   **Hallucinations:**
    *   **Topic 5 (Digital Divide):** The keywords describe *synthetic media* (content generation), but Qwen labeled it **"Digital Divide"** (access gap). This is a semantic hallucination, likely driven by the model conflating "technology issues."
    *   **Topic 8 (Trust in the Long Tail):** Keywords (`trust`, `constituents`) suggest stakeholder engagement, but the label **"Trust in the Long Tail"** invents a statistical concept not present in the text.

### Australia: Refining "Practitioner Discourse"
LDA characterized Australia through "practitioner-led" surveys and "cognitive science." The new model refines this by isolating theoretical frameworks:
*   **Granularity Gain:** It confirms the **"Cognitive Offloading"** (Topic 1) theme with high precision, validating the LDA finding. Furthermore, it splits the broad "submission" theme into **"Academic and Institutional Support"** (Topic 6), highlighting the tertiary/adoption phase specific to the Australian corpus.
*   **New Topic - Theoretical Literacy:** Topic 3 (`literacy`, `computational`, `studies`) is labeled **"Theoretical and Practical Approaches,"** successfully capturing the dual nature of the literacy debate (theory vs. practice) that LDA treated as a single keyword blob.
*   **Hallucinations:**
    *   **Topic 2 (Workforce):** The keywords (`respondents`, `survey`, `independent`) clearly indicate *public consultation*. However, Qwen labeled this **"Workforce."** This misinterpretation likely stems from the presence of employment-related terms in the survey responses, missing the "consultation" context entirely.

The **Qwen/Qwen2.5-0.5B-Instruct** model proves effective at rapid, concise labeling, successfully identifying niche themes like **"Synthetic Media"** and **"Staff Privacy"** that LDA obscured. However, the analysis demonstrates that **SLMs are prone to hallucination when constrained by strict word counts** (2-3 words). When the semantic signal in the keywords is ambiguous (e.g., "survey" vs. "workforce"), the model tends to default to high-probability policy buzzwords like "Digital Divide" or "Workforce," creating a disconnect between the underlying data and the assigned label.